# RetinAI — 5-Fold Cross Validation Eğitimi

**Yöntem:** 5-Fold Cross Validation  
**Modeller:** UNet, Attention U-Net, ResUNet, SegFormer, Swin-UNet  
**Veri:** Test seti dışarıda tutulur, ~480 görüntü K-Fold'a girer

> ✅ **Runtime → Change runtime type → T4 GPU** seçili olduğundan emin ol!

## Adım 1 — Google Drive'ı Bağla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Drive'daki proje klasörünün yolunu ayarla
# Örnek: Drive'ında 'retina_project' klasörü varsa:
PROJECT_PATH = '/content/drive/MyDrive/retina_project'

import os
print('Proje klasörü var mı:', os.path.exists(PROJECT_PATH))
print(os.listdir(PROJECT_PATH))

## Adım 2 — Bağımlılıkları Kur

In [ ]:
!pip install -q opencv-python-headless torch torchvision

## Adım 3 — Proje Dizinine Geç

In [ ]:
import sys
os.chdir(PROJECT_PATH)
sys.path.insert(0, os.path.join(PROJECT_PATH, 'src'))

import torch
print('PyTorch versiyonu:', torch.__version__)
print('CUDA kullanılabilir:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Adım 4 — Veri Klasörlerini Kontrol Et

In [ ]:
IMG_DIR  = os.path.join(PROJECT_PATH, 'data', 'processed', 'drive', 'images_png')
MASK_DIR = os.path.join(PROJECT_PATH, 'data', 'processed', 'drive', 'masks_png')
TEST_IDS = os.path.join(PROJECT_PATH, 'data', 'test_ids.txt')

print('Görüntü klasörü:', os.path.exists(IMG_DIR), '—', len(os.listdir(IMG_DIR)) if os.path.exists(IMG_DIR) else 'YOK')
print('Maske klasörü  :', os.path.exists(MASK_DIR), '—', len(os.listdir(MASK_DIR)) if os.path.exists(MASK_DIR) else 'YOK')
print('Test ID dosyası:', os.path.exists(TEST_IDS))

## Adım 5 — Model Seç ve K-Fold Başlat

Her modeli sırayla çalıştır. Bir model bitince sonraki hücreye geç.

In [ ]:
# ── ResUNet (Ana Model) ──────────────────────────────────────────
%run src/train_kfold.py --model resunet

In [ ]:
# ── UNet ────────────────────────────────────────────────────────
%run src/train_kfold.py --model unet

In [ ]:
# ── Attention U-Net ─────────────────────────────────────────────
%run src/train_kfold.py --model attention_unet

In [ ]:
# ── SegFormer Lite ──────────────────────────────────────────────
%run src/train_kfold.py --model segformer

In [ ]:
# ── Swin-UNet ───────────────────────────────────────────────────
%run src/train_kfold.py --model swinunet

## Adım 6 — K-Fold Sonuçlarını Görüntüle

In [ ]:
import pandas as pd

summary_path = os.path.join(PROJECT_PATH, 'results', 'logs', 'kfold_summary.csv')
if os.path.exists(summary_path):
    df = pd.read_csv(summary_path)
    print('\n=== 5-Fold Cross Validation Sonuçları ===')
    print(df.to_string(index=False))
    print('\nEn iyi model (Mean Dice):', df.loc[df['mean_dice'].idxmax(), 'model'])
else:
    print('Henüz sonuç yok, önce eğitimi tamamla.')

## Adım 7 — Test Seti Değerlendirmesi

In [ ]:
# K-Fold'da en iyi ağırlıklar results/models/kfold/ klasörüne kaydedildi.
# evaluate.py normal models/ klasörüne bakıyor;
# önce en iyi fold modelini oraya kopyala:

import shutil

# Örnek: ResUNet için en iyi fold modelini kopyala
# (kfold_summary.csv'ye bakarak hangi fold'un en iyi olduğunu bul)
kfold_dir  = os.path.join(PROJECT_PATH, 'results', 'models', 'kfold')
models_dir = os.path.join(PROJECT_PATH, 'results', 'models')

model_fold_map = {
    'resunet':        1,   # kfold_summary.csv'ye göre güncellenebilir
    'unet':           1,
    'attention_unet': 1,
    'segformer':      1,
    'swinunet':       1,
}

for model_name, best_fold in model_fold_map.items():
    src  = os.path.join(kfold_dir,  f'{model_name}_fold{best_fold}_best.pth')
    dst  = os.path.join(models_dir, f'{model_name}_best.pth')
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'Kopyalandı: {model_name} fold{best_fold} → models/{model_name}_best.pth')
    else:
        print(f'UYARI: {src} bulunamadı')

In [ ]:
# Test seti üzerinde final değerlendirme
%run src/evaluate.py

## Adım 8 — Sonuçları Drive'a Kaydet

In [ ]:
# Colab oturumu kapanmadan önce sonuçları Drive'a zaten kaydoluyor
# (proje zaten Drive'da). Kontrol et:

results_dir = os.path.join(PROJECT_PATH, 'results')
for root, dirs, files in os.walk(results_dir):
    for f in files:
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / (1024*1024)
        print(f'{os.path.relpath(fpath, PROJECT_PATH):60s}  {size_mb:.1f} MB')